In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv('../.env')

DB_HOST = os.getenv('DB_HOST')
DB_PORT = 3306
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

In [ ]:
DATE = '2026-01-23'
PLATFORM = 1
INCIDENT_TYPES = ['later_than_following_tram', 'cancelled', 'major_delay']

SQL_QUERY = f"""
    SELECT datetime, realdatetime, delay FROM kruppallee.departures 
    WHERE
        DATE(datetime) = '{DATE}'
        AND line in ('107', '108')
        AND platform = {PLATFORM}
    ORDER BY datetime ASC
"""

# Build SQLAlchemy engine for MySQL
engine = create_engine(
    f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_DATABASE}"
)

df = pd.read_sql(SQL_QUERY, con=engine)
df['time'] = pd.to_datetime(df['datetime']).dt.time
df['realtime'] = pd.to_datetime(df['realdatetime']).dt.time
df.drop(columns=['datetime', 'realdatetime'], inplace=True)
df.head()

In [ ]:
df1 = df[df['realtime'].notna()]
df1 = df1.reset_index(drop=True)

df1['realindex'] = df1['realtime'].rank(method='first').astype(int) - 1
df2 = df1[df1.index != df1['realindex']]
df2

In [ ]:
df3 = df[df['delay'] >= 10]
df3